# CIFAR-10-C / CIFAR-100-C Robustness Evaluation

Evaluates **Teacher / Student-no-KD / KL-KD / Fixed-OT-KD / Adaptive-OT-KD** on the official Hendrycks & Dietterich (2019) corruption benchmarks. Default corruptions:

- `gaussian_noise` — additive sensor noise
- `gaussian_blur` — out-of-focus blur
- `spatter` — splotchy droplet occlusion

Each runs at 5 severity levels. Numbers are directly comparable to published CIFAR-C results.

> **Heads up:** the official CIFAR-10-C tar is **2.9 GB** and CIFAR-100-C is another **2.9 GB**. These download once into Colab's local disk, get extracted, and the tar is then auto-deleted to save space. End-to-end runtime on a T4 GPU is ~10 min for both datasets; on a CPU runtime, ~30 min.

## 0. GPU runtime check
If you don't see your GPU below, switch via **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi || echo 'No GPU — Runtime > Change runtime type > GPU'

## 1. Upload the project zip

This notebook is self-contained — it expects the `sinkhorn-vision-kd` project (source files **and** the `*_best.pth` / `*_teacher.pth` checkpoints under `checkpoints/cifar10/` and `checkpoints/cifar100/`) to be uploaded as a single zip.

Run the cell, click **Choose Files**, and pick your `sinkhorn-vision-kd.zip`.

In [ ]:
%cd /content
from google.colab import files
import os, zipfile
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn) as zf:
            zf.extractall('./')
        os.remove(fn)
if os.path.isdir('sinkhorn-vision-kd'):
    %cd sinkhorn-vision-kd
print(sorted(os.listdir('.')))

## 2. Status check
Skip running this if you already know they're there. If anything reports `MISS`, fix it before downloading 5 GB of CIFAR-C.

In [ ]:
import os
all_ok = True
for ds in ['cifar10', 'cifar100']:
    d = f'./checkpoints/{ds}'
    print(f'\n[{ds}]  ({d})')
    if not os.path.isdir(d):
        print('   MISSING DIR'); all_ok = False; continue
    needed = [
        f'{ds}_resnet110_teacher.pth',
        'resnet20_no_kd_best.pth',
        'kl_kd_best.pth',
        'sinkhorn_kd_best.pth',
        'adaptive_sinkhorn_kd_best.pth',
    ]
    for f in needed:
        p = os.path.join(d, f)
        print(('   OK   ' if os.path.exists(p) else '   MISS '), f)
        if not os.path.exists(p): all_ok = False
print('\nALL GOOD' if all_ok else '\nSOME CHECKPOINTS MISSING — see above')

## 3. CIFAR-10-C robustness sweep
First run downloads `CIFAR-10-C.tar` (2.9 GB) from Zenodo, extracts the three corruption files we need, then deletes the tar.

In [ ]:
!python evaluate_robustness.py \
    --dataset cifar10 \
    --batch_size 512 \
    --num_workers 2 \
    --save_csv robustness_cifar10.csv

## 4. CIFAR-100-C robustness sweep
Downloads `CIFAR-100-C.tar` (2.9 GB) the first time.

In [ ]:
!python evaluate_robustness.py \
    --dataset cifar100 \
    --batch_size 512 \
    --num_workers 2 \
    --save_csv robustness_cifar100.csv

## 5. Pretty-print the CSVs

In [ ]:
import os, pandas as pd
for f in ['robustness_cifar10.csv', 'robustness_cifar100.csv']:
    if not os.path.exists(f):
        continue
    print(f'\n=== {f} ===')
    df = pd.read_csv(f)
    pivot = df.pivot_table(index='method', columns=['corruption', 'severity'],
                           values='top1_acc')
    print(pivot.round(2).to_string())

## 6. Plot accuracy vs severity

In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt

for ds in ['cifar10', 'cifar100']:
    csv = f'robustness_{ds}.csv'
    if not os.path.exists(csv):
        continue
    df = pd.read_csv(csv)
    df_corr = df[df.corruption != 'clean']
    corruptions = sorted(df_corr.corruption.unique())
    fig, axes = plt.subplots(1, len(corruptions),
                             figsize=(5*len(corruptions), 4), sharey=True)
    if len(corruptions) == 1: axes = [axes]
    for ax, corr in zip(axes, corruptions):
        sub = df_corr[df_corr.corruption == corr]
        for method, g in sub.groupby('method'):
            g = g.sort_values('severity')
            ax.plot(g.severity, g.top1_acc, marker='o', label=method)
        ax.set_title(f'{ds} — {corr}')
        ax.set_xlabel('severity'); ax.set_ylabel('top-1 acc (%)')
        ax.grid(alpha=0.3)
    axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout(); plt.show()

## 7. Download CSVs

In [ ]:
from google.colab import files
import os
for f in ['robustness_cifar10.csv', 'robustness_cifar100.csv']:
    if os.path.exists(f):
        files.download(f)